In [ ]:
MONGODB_URI = ("HAHA")

In [ ]:
import os
from pymongo import MongoClient
from sentence_transformers import SentenceTransformer

# Connect to your local MongoDB (change this if needed)
client = MongoClient(MONGODB_URI)

# Select the collection
collection = client["CovidDB"]["CovidCases"]

# Get the current working directory
notebook_dir = os.getcwd()
model_path = os.path.join(notebook_dir, "mxbai-embed-large-v1")

# Load the embedding model
model = SentenceTransformer('mixedbread-ai/mxbai-embed-large-v1')

def get_embedding(text):
    return model.encode(text).tolist()

# Filter: only documents without an 'embeddings' field
filter = { 'embeddings': { '$exists': False } }

# Update documents
updated_doc_count = 0
for document in collection.find(filter).limit(50): #just first 50 for now
    # Remove the '_id' key so it's not embedded
    doc_to_embed = {k: v for k, v in document.items() if k != '_id'}
    
    # Convert the full document to a string
    doc_text = str(doc_to_embed)

    # Generate and store the embedding
    embedding = get_embedding(doc_text)
    collection.update_one({ '_id': document['_id'] }, { "$set": { 'embeddings': embedding } }, upsert=True)
    updated_doc_count += 1

print(f"Documents updated: {updated_doc_count}")


c:\Users\towas\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\towas\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\towas\.cache\huggingface\hub\models--mixedbread-ai--mxbai-embed-large-v1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Py

### Use Atlas Vector Search to retrieve relevant documents.

In [ ]:
# Function to get the results of a vector search query
def get_query_results(query):
   query_embedding = get_embedding(query)

   pipeline = [
      {
            "$vectorSearch": {
               "index": "vector_index",
               "queryVector": query_embedding,
               "path": "embeddings",
               "exact": True,
               "limit": 5
            }
      }, {
            "$project": {
               "_id": 0,
               "summary": 1,
               "listing_url": 1,
               "score": {
                  "$meta": "vectorSearchScore"
               }
            }
      }
   ]

   results = collection.aggregate(pipeline)

   array_of_results = []
   for doc in results:
      array_of_results.append(doc)
   return array_of_results

# TESTING EXISTENCE OF EMEBDDINE AND if it returns query results in coordinates

In [ ]:
import pprint
pprint.pprint(get_query_results("ICU"))

[{'score': 0.8028285503387451},
 {'score': 0.8014934659004211},
 {'score': 0.8003019690513611},
 {'score': 0.8000949621200562},
 {'score': 0.7995988130569458}]


In [ ]:
from gpt4all import GPT4All

local_llm_path = "./mistral-7b-openorca.gguf2.Q4_0.gguf"
local_llm = GPT4All(local_llm_path)

Downloading: 100%|██████████| 4.11G/4.11G [02:27<00:00, 27.9MiB/s]


In [ ]:
question = "What are common sources of infection in Mount Pleasant West?"

documents = get_query_results(question)

text_documents = ""
for doc in documents:
    parts = []
    for k, v in doc.items():
        if k != '_id' and k != 'embeddings':
            parts.append(f"{k}: {v}")
    string = " | ".join(parts)
    text_documents += string + "\n"

prompt = f"""Use the following pieces of context to answer the question at the end.
{text_documents}
Question: {question}
"""

response = local_llm.generate(prompt)
cleaned_response = response.replace('\\n', '\n')
print(cleaned_response)



Answer: Common sources of infection in Mount Pleasant West include contaminated water, poor sanitation, and overcrowded living conditions. These factors can lead to the spread of various diseases and contribute to a lower overall health score for the area.
